In [3]:
from config import CONFIG
from modes import build_runtime_mode_by_name
from workloads import build_runtime_workload_by_name
from model_loader import load_model_for_mode, unload_model
from runner import run_single_benchmark

In [4]:
from huggingface_hub import login
login(new_session=True)

In [5]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '695f984390fca78c745ec290', 'name': 'Aman-Sunesh', 'fullname': 'Aman Sunesh', 'email': 'as18181@nyu.edu', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/fpZG3qyPrGgXiW-TopC8U.jpeg', 'orgs': [{'type': 'org', 'id': '691d8e884bbe8df0d99462e2', 'name': 'newyorkuniversity', 'fullname': 'New York University', 'email': 'islam.m@nyu.edu', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/68e396f2b5bb631e9b2fac9a/orNHmPzOQf2_F5UgXPsu5.png', 'roleInOrg': 'write'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'ModeSwitch-LLM', 'role': 'read', 'createdAt': '2026-04-11T22:53:44.893Z'}}}


In [6]:
print("Model:", CONFIG.model.model_name_or_path)
print("Backend:", CONFIG.model.backend)
print("Device:", CONFIG.model.device)
print("Baseline dtype:", CONFIG.model.baseline_dtype)
print("Trials:", CONFIG.system.num_trials)

Model: meta-llama/Llama-3.1-8B-Instruct
Backend: transformers
Device: cuda
Baseline dtype: float16
Trials: 3


In [7]:
runtime_mode = build_runtime_mode_by_name("fp16_baseline")
runtime_workload = build_runtime_workload_by_name("short_prompt_short_output")

print("Mode:")
print(runtime_mode)

print("\nWorkload:")
print(runtime_workload.name)
print("Max new tokens:", runtime_workload.max_new_tokens)
print("Prompt preview:")
print(runtime_workload.prompt[:300])

Mode:
RuntimeMode(name='fp16_baseline', description='FP16 baseline inference mode', backend='transformers', dtype='float16', quantization=None, primary_phase='both', notes='standard mode', speculative_decoding=False, kv_cache_compression=False, prefix_caching=False, chunked_prefill=False, continuous_batching=False, cuda_graphs=False, runtime_kwargs={'torch_dtype': 'float16'})

Workload:
short_prompt_short_output
Max new tokens: 32
Prompt preview:
You are a helpful assistant. Answer the following question clearly and concisely:

Explain the importance of efficient inference for large language models.

You are a helpful assistant. Answer the following question clearly and concisely:

Explain the importance of efficient inference for large lang


In [8]:
bundle = None

try:
    bundle = load_model_for_mode(runtime_mode)
    print("Model loaded successfully.")
    print("Mode name:", bundle.mode_name)
    print("Backend:", bundle.backend)
    print("Tokenizer type:", type(bundle.tokenizer))
    print("Model type:", type(bundle.model))
except Exception as e:
    print("LOAD FAILED:")
    print(type(e).__name__, str(e))
finally:
    unload_model(bundle)

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\amans\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amans\.cache\huggingface\hub\models--meta-llama--Llama-3.1-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully.
Mode name: fp16_baseline
Backend: transformers
Tokenizer type: <class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>
Model type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


In [9]:
result = run_single_benchmark(
    runtime_mode=runtime_mode,
    workload=runtime_workload,
    trial_index=0,
)

print(result)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BenchmarkResult(mode_name='fp16_baseline', workload_name='short_prompt_short_output', backend='transformers', trial_index=0, prompt_tokens_target=128, max_new_tokens=32, repeated_prefix=False, memory_pressure=False, start_time_s=277882.9910212, first_token_time_s=277882.9957667, end_time_s=277913.4583153, token_timestamps_s=[], output_text='  </s>\n\nEfficient inference is crucial for large language models as it enables them to process and respond to user queries quickly and accurately. Large language models are', output_tokens_generated=32, ttft_ms=4.74549998762086, avg_tbt_ms=None, total_latency_ms=30467.294099973515, decode_latency_ms=30462.548599985894, tokens_per_second=1.0503065974614338, peak_gpu_memory_mb=15372.90673828125, reserved_gpu_memory_mb=15516.0, avg_power_w=None, energy_joules=None, energy_per_token_j=None, notes='', error=None)


In [10]:
result_dict = result.to_dict()

for k, v in result_dict.items():
    print(f"{k}: {v}")

mode_name: fp16_baseline
workload_name: short_prompt_short_output
backend: transformers
trial_index: 0
prompt_tokens_target: 128
max_new_tokens: 32
repeated_prefix: False
memory_pressure: False
start_time_s: 277882.9910212
first_token_time_s: 277882.9957667
end_time_s: 277913.4583153
token_timestamps_s: []
output_text:   </s>

Efficient inference is crucial for large language models as it enables them to process and respond to user queries quickly and accurately. Large language models are
output_tokens_generated: 32
ttft_ms: 4.74549998762086
avg_tbt_ms: None
total_latency_ms: 30467.294099973515
decode_latency_ms: 30462.548599985894
tokens_per_second: 1.0503065974614338
peak_gpu_memory_mb: 15372.90673828125
reserved_gpu_memory_mb: 15516.0
avg_power_w: None
energy_joules: None
energy_per_token_j: None
notes: 
error: None


In [11]:
if result.error is not None:
    print("Run failed.")
    print(result.error)
else:
    print("Run succeeded.")
    print("TTFT (ms):", result.ttft_ms)
    print("Avg TBT (ms):", result.avg_tbt_ms)
    print("Total latency (ms):", result.total_latency_ms)
    print("Peak GPU memory (MB):", result.peak_gpu_memory_mb)
    print("Output tokens:", result.output_tokens_generated)
    print("Output text preview:")
    print((result.output_text or "")[:500])

Run succeeded.
TTFT (ms): 4.74549998762086
Avg TBT (ms): None
Total latency (ms): 30467.294099973515
Peak GPU memory (MB): 15372.90673828125
Output tokens: 32
Output text preview:
  </s>

Efficient inference is crucial for large language models as it enables them to process and respond to user queries quickly and accurately. Large language models are
